# VectorCraft Scripting

Example notebook for the Python bridge to VectorCraft's headless CLI (`vectorcraft-cli`): place Plotly/SVG figures into a document and export PDF/SVG/`.vectorcraft`.

Requires VectorCraft installed (Windows default: `C:\\Program Files\\VectorCraft\\vectorcraft-cli.exe`).

In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

import sys
from copy import deepcopy
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
from collections import defaultdict
from functools import partial
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)

import IPython
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from pyphocorehelpers.programming_helpers import IPythonHelpers
from pyphocorehelpers.assertion_helpers import Assert
from pyphocorehelpers.print_helpers import print_keys_if_possible, print_value_overview_only
from pyphocorehelpers.preferences_helpers import set_pho_preferences
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path, file_uri_from_path, sanitize_filename_for_Windows
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget
from pyphocorehelpers.pho_jupyter_preview_widget.display_helpers import path_repr_with_filesystem_link

set_pho_preferences(is_concise=False)

_notebook_path: Path = Path(IPythonHelpers.try_find_notebook_filepath(IPython.extract_module_locals())).resolve()
_notebook_dir: Path = _notebook_path.parent
_package_root: Path = _notebook_dir.parent
_package_src: Path = _package_root.joinpath('src').resolve()
if str(_package_src) not in sys.path:
    sys.path.insert(0, str(_package_src))

ip = IPython.get_ipython()
ip = path_repr_with_filesystem_link(ip=ip)

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
template: str = 'plotly_dark'
pio.templates.default = template

from vectorcraft_scripting import (
    CliNotFoundError,
    PlaceSpec,
    VectorCraftClient,
    VectorCraftError,
    export_plotly_figure,
    place_into_template,
    write_plotly_svg,
)

print(f'_notebook_path: {_notebook_path}')
print(f'_package_src: {_package_src}')

Automatic pdb calling has been turned OFF
_notebook_path: H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\VectorCraft_Scripting\notebooks\VectorCraft_Scripting.ipynb
_package_src: H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\VectorCraft_Scripting\src


## Paths / output folder

In [2]:
outputs_folder: Path = _notebook_dir.joinpath('outputs').resolve()
outputs_folder.mkdir(parents=True, exist_ok=True)
display(fullwidth_path_widget(outputs_folder, file_name_label='VectorCraft outputs:'))
print(f'outputs_folder: {outputs_folder}')
print(f'file_uri: {file_uri_from_path(outputs_folder)}')

Box(children=(Label(value='VectorCraft outputs:', layout=Layout(width='auto')), HTML(value="<b style='font-siz…

outputs_folder: H:\TEMP\Spike3DEnv_ExploreUpgrade\Spike3DWorkEnv\Spike3D\EXTERNAL\VectorCraft_Scripting\notebooks\outputs
file_uri: file:///H:/TEMP/Spike3DEnv_ExploreUpgrade/Spike3DWorkEnv/Spike3D/EXTERNAL/VectorCraft_Scripting/notebooks/outputs


## Discover `vectorcraft-cli` (Windows)

In [3]:
vc = VectorCraftClient()  # VECTORCRAFT_CLI → PATH → C:\\Program Files\\VectorCraft\\vectorcraft-cli.exe
print(f'cli_path: {vc.cli_path}')
Assert.path_exists(vc.cli_path)

catalogue: List[Dict[str, Any]] = vc.commands()
print(f'n_commands: {len(catalogue)}')
print(f'first: {catalogue[0]["id"]}')

place_docs = next((c for c in catalogue if c.get('id') == 'file.place'), None)
if place_docs is not None:
    print(f"file.place enabled={place_docs.get('enabled')} label={place_docs.get('label')!r}")
    print(str(place_docs.get('params', ''))[:500], '...')

cli_path: C:\Program Files\VectorCraft\vectorcraft-cli.exe
n_commands: 785
first: file.new
file.place enabled=False label='Place…'
{path | name | dataBase64 (a file's path, or its name with its bytes), link?: true (a raster image keeps its file's path; a VectorCraft document at path becomes one locked placed document showing artboard `page`, linked to the file and read again when it changes (links.update; saving it in this app updates it at once), drawn as vectors in every output; false: an editable copy of its art, as from dataBase64; other files are embedded), text?: {characterSet?: "unicode" (UTF-8, or UTF-16 with a byte ...


## Smoke: `file.new` → export SVG

In [ ]:
smoke_svg: Path = outputs_folder.joinpath('smoke_blank.svg').resolve()
smoke_steps = vc.run(
    commands=[('file.new', {'width': 560, 'height': 360})],
    exports=[smoke_svg],
)
print_value_overview_only(smoke_steps)
Assert.path_exists(smoke_svg)
display(fullwidth_path_widget(smoke_svg, file_name_label='smoke SVG:'))

## Plotly figure → SVG → place → export

Creates a new document, places the chart in a rect (document points: `[x, y, width, height]` from the artboard top-left), and writes `.vectorcraft` + PDF.

In [ ]:
fig = go.Figure(go.Scatter(x=[1, 2, 3], y=[2, 1, 3], mode='lines+markers', name='demo'))
fig.update_layout(width=560, height=360, margin=dict(l=40, r=16, t=32, b=32), title='VectorCraft place demo')
fig.show()

chart_svg: Path = outputs_folder.joinpath('chart.svg').resolve()
write_plotly_svg(fig, chart_svg)
Assert.path_exists(chart_svg)

# 36 pt from left, 80 pt from top, 280×180 pt box
place = PlaceSpec(path=chart_svg, rect=[36, 80, 280, 180], crop='crop', link=False)

out_vc: Path = outputs_folder.joinpath('out.vectorcraft').resolve()
out_pdf: Path = outputs_folder.joinpath('out.pdf').resolve()

place_steps = vc.run(
    commands=[
        ('file.new', {'width': 612, 'height': 792}),  # letter-ish artboard
        place.as_command(),
    ],
    exports=[out_vc, out_pdf],
)
print_keys_if_possible('place_steps', place_steps, max_depth=2)
Assert.path_exists(out_vc)
Assert.path_exists(out_pdf)
display(fullwidth_path_widget(out_pdf, file_name_label='placed PDF:'))

## Optional: place into an existing `.vectorcraft` template

If you have a layout template, set `template_path` and run the cell below (same as the README `export_plotly_figure` / `place_into_template` helpers).

In [ ]:
template_path: Optional[Path] = None
# template_path = Path(r'H:/path/to/template.vectorcraft').resolve()

if template_path is None:
    print('Set template_path to a .vectorcraft file to run this cell.')
else:
    Assert.path_exists(template_path)
    templated_pdf: Path = outputs_folder.joinpath('from_template.pdf').resolve()
    results = export_plotly_figure(
        fig,
        chart_svg=outputs_folder.joinpath('chart_for_template.svg'),
        template=template_path,
        rect=[36, 480, 280, 180],
        exports=[templated_pdf],
        client=vc,
    )
    print_value_overview_only(results)
    display(fullwidth_path_widget(templated_pdf, file_name_label='template PDF:'))

## Inspect a document

In [ ]:
summary = vc.info(out_vc)
print_keys_if_possible('info', summary, max_depth=2)
summary.get('artboards')